In [1]:

import argparse
import logging
from types import SimpleNamespace
import os
from pathlib import Path
import csv
import sys
import math
import numpy as np
import pandas as pd
import time
import joblib
import json 
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# sys.path.append('/Users/yiqin/Documents/PythonProjects/GraspDataProcessing/src')
# sys.path.append('D:\\PythonPrograms\\GraspDataProcessing\\src')
sys.path.append('D:\\PythonProjects\\GraspDataProcessing\\src')
import graspdataprocessing as gdp


In [2]:
config_file = 'config.toml'  # 配置文件的路径
config = gdp.load_config(config_file)

配置验证通过: cutoff_value=1e-09, initial_ratio=0.09


In [4]:
"""主程序逻辑"""
config.file_name = f'{config.conf}_{config.cal_loop_num}'
logger = gdp.setup_logging(config)
logger.info("机器学习训练程序启动")
execution_time = time.time()

gdp.setup_directories(config)
# 初始化结果文件
# gdp.initialize_results_file(config, logger)

# 验证初始文件
gdp.validate_initial_files(config, logger)

logger.info(f"初始比例: {config.initial_ratio}")
logger.info(f"光谱项: {config.spetral_term}")

2025-06-18 21:37:21,733 - INFO - 机器学习训练程序启动
2025-06-18 21:37:21,734 - INFO - 成功加载初始CSFs文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.c
2025-06-18 21:37:21,734 - INFO - 初始比例: 0.09
2025-06-18 21:37:21,735 - INFO - 光谱项: ['5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D', '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D']


In [10]:
try:
    # 加载数据文件
    energy_level_data_pd, rmix_file_data, target_pool_csfs_data, raw_csfs_descriptors, cal_csfs_data, caled_csfs_indices_dict = gdp.load_data_files(config, logger)
    
    # 检查组态耦合
    cal_result, asfs_position = gdp.check_configuration_coupling(config, energy_level_data_pd, logger)
    logger.info("************************************************")

except Exception as e:
        logger.error(f"程序执行过程中发生错误: {str(e)}")
        raise

2025-06-18 21:40:15,050 - INFO - 加载能级数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level


Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
data file type: level data
energy levels data has been written into LEVEL/cv4odd1as3_odd4_1.level_level.csv csv file
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m loaded.
data file type: rmcdhf mix_coefficient_data
g92mix: G92MIX
 nblock = 1,       ncftot =   385600,          nw =  41,            nelec =   64


100%|██████████| 1/1 [00:00<00:00, 1000.31it/s]
2025-06-18 21:40:15,054 - INFO - 加载 *.m 文件数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m


cycle jblock = 1
 Block no. = 1, 2J+1 = 9, Parity = -1, No. of eigenvalues = 2, No. of CSFs = 385600

    Energy levels for ...
Rydberg constant is  109737.31568508

---------------------------------------------
 No Pos  J  Parity   Energy Total    Levels
                      (a.u.)         (cm^-1)
---------------------------------------------

  1  1   4   +    -11274.7413433        0.00
  2  0   4   +    -11274.7204780     4579.40


2025-06-18 21:40:20,933 - INFO - 加载初始 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.pkl.gz
2025-06-18 21:40:21,162 - INFO - 加载初始 CSFs 描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4


Descriptors loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors.npy
Labels loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors_block_indices.npy
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c loaded.


2025-06-18 21:40:21,598 - INFO - 加载本轮计算 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c
2025-06-18 21:40:21,610 - INFO - 加载本轮选择的 CSFs 的索引文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_chosen_indices.pkl
2025-06-18 21:40:22,000 - INFO - 光谱项 '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D' 在位置 0 找到
2025-06-18 21:40:22,001 - INFO - 光谱项 '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D' 在位置 1 找到
2025-06-18 21:40:22,002 - INFO - cal_loop 1 组态耦合正确，位置索引: [0, 1]
2025-06-18 21:40:22,002 - INFO - ************************************************


In [11]:
logger.info("             数据预处理")
caled_csfs_descriptors = gdp.generate_chosen_csfs_descriptors(config, caled_csfs_indices_dict, raw_csfs_descriptors, rmix_file_data, asfs_position, logger)
unselected_csfs_descriptors = gdp.get_unselected_descriptors(raw_csfs_descriptors, caled_csfs_indices_dict)
X_unselected = unselected_csfs_descriptors.copy()
logger.info("             特征提取完成")

2025-06-18 21:40:23,895 - INFO -              数据预处理


2025-06-18 21:40:24,059 - INFO - 保存本轮选择的 CSFs 的描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1/cv4odd1as3_odd4_1.npy


Descriptors saved to: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_descriptors.npy


2025-06-18 21:40:24,865 - INFO -              特征提取完成


In [12]:
model, X_train, X_test, y_train, y_test, training_time, weight = gdp.train_model(config, caled_csfs_descriptors, rmix_file_data, logger)

2025-06-18 21:40:29,169 - INFO - 训练集 - 正样本:22038, 负样本:286442, 比例:0.0714
2025-06-18 21:40:30,279 - INFO - 创建新模型，设置类别权重: 负样本=1.0, 正样本=13.0
2025-06-18 21:40:30,280 - INFO - 使用原始数据训练 - 不进行重采样
2025-06-18 21:40:30,280 - INFO - 最终训练数据 - 正样本:22038, 负样本:286442, 比例:0.0714
2025-06-18 21:40:30,281 - INFO - 使用类别权重和损失函数来处理数据不平衡问题
2025-06-18 21:40:30,281 - INFO -              训练模型
2025-06-18 21:40:30,281 - INFO - 开始训练 - 数据量:308,480, 特征维度:87
2025-06-18 21:40:45,809 - INFO - Epoch [50/150], Loss: 0.327835
2025-06-18 21:41:00,505 - INFO - Epoch [100/150], Loss: 0.314873
2025-06-18 21:41:15,340 - INFO - Epoch [150/150], Loss: 0.307942
2025-06-18 21:41:15,341 - INFO - 训练Loss良好 (0.310)，模型收敛效果理想
2025-06-18 21:41:15,342 - INFO -              预测与评估
2025-06-18 21:41:15,439 - INFO - 预测概率统计 - 最小值:0.0000, 最大值:1.0000, 平均值:0.2528
2025-06-18 21:41:15,440 - INFO - 预测为正类的样本数: 5584/77120
2025-06-18 21:41:15,440 - INFO - 真实正样本数: 5423.0/77120
2025-06-18 21:41:15,441 - INFO - 真实正样本比例: 0.070, 预测正样本比例: 0.072


(385600,)


2025-06-18 21:41:17,031 - INFO - 测试集预测结果:
2025-06-18 21:41:17,032 - INFO - AUC:0.8812714130825584, pr_auc:0.7686726562882413, f1:0.749159625692741, accuracy:0.9641986514522821, precision:0.7383595988538681, recall:0.7602802876636549
2025-06-18 21:41:17,130 - INFO - 训练集预测结果:
2025-06-18 21:41:17,131 - INFO - AUC:0.9463473182886589, f1:0.7695810975334806, accuracy:0.9668698132780082, precision:0.7647875963434307, recall:0.7744350667029676


In [13]:
evaluation_results = gdp.evaluate_model(
            model, X_train, X_test, y_train, y_test, X_unselected, config, logger
        )

2025-06-18 21:41:25,283 - INFO - 开始预测与评估
2025-06-18 21:41:26,217 - INFO - 模型评估完成


In [14]:
test_predictions = evaluation_results['predictions']['y_prediction_test']
test_probabilities = evaluation_results['probabilities']['y_probability_test']
test_f1 = evaluation_results['test_metrics']['f1']
train_f1 = evaluation_results['train_metrics']['f1']

In [15]:
overfitting_check = test_f1 - train_f1  # 如果差异过大说明过拟合

In [16]:
overfitting_check

-0.020421471840739613

In [17]:
logger.info("             模型推理")
start_time = time.time()
X_unselected = unselected_csfs_descriptors.copy()
y_unselected_prediction = model.predict(X_unselected)
y_unselected_probability = model.predict_probability(X_unselected)[:, 1]
eval_time = time.time() - start_time
logger.info(f"             模型推理时间:{eval_time}")

2025-06-18 21:41:38,705 - INFO -              模型推理
2025-06-18 21:41:39,546 - INFO -              模型推理时间:0.8414239883422852


In [63]:
chosen_indiceasdga = gdp.csfs_index_load('D:\\PythonProjects\\as3_odd4\\cv4odd1as3_odd4_1\\cv4odd1as3_odd4_1_chosen_indices.pkl')

In [62]:
unchosen_indiceasdga = gdp.csfs_index_load('D:\\PythonProjects\\as3_odd4\\cv4odd1as3_odd4_1\\cv4odd1as3_odd4_1_unselected_indices.pkl')

In [66]:
len(chosen_indiceasdga[0])

385600

In [67]:
len(unchosen_indiceasdga[0])


3898846

In [27]:
y_prediction = evaluation_results['predictions']['y_prediction_test']
y_probability = evaluation_results['probabilities']['y_probability_test']
result_file_path = config.root_path / 'test_data' / f'{config.conf}_{config.cal_loop_num}.csv'
pd.DataFrame({"y_test": y_test, "y_prediction": y_prediction, "y_probability": y_probability}).to_csv(result_file_path, index=False)
model_file_path = config.root_path / 'models' / f'{config.conf}_{config.cal_loop_num}.pkl'
joblib.dump(model, model_file_path)
logger.info(f"             预测结果与模型保存成功")

csfs_above_threshold_indices = np.where(np.any(rmix_file_data.mix_coefficient_List[0][asfs_position]**2 >= np.float64(config.cutoff_value), axis = 0))[0]

ml_chosen_from_unselected_indices = np.where(y_unselected_prediction == 1)[0]

2025-06-18 21:46:15,780 - INFO -              预测结果与模型保存成功


In [37]:
y_probability_all = evaluation_results['probabilities']['y_probability_all']

In [43]:
y_probability_other = evaluation_results['probabilities']['y_probability_other']


In [47]:
high_prob_threshold = np.percentile(y_probability_all[:, 1], 90)  # 取90分位数作为高概率阈值

In [49]:
promising_unselected = X_unselected[y_probability_other > high_prob_threshold]

In [53]:
high_prob_indices = np.where(y_probability_other > high_prob_threshold)[0]

In [54]:
high_prob_indices

array([     14,      27,     135, ..., 3898743, 3898781, 3898811])

In [50]:
promising_unselected

array([[2., 0., 0., ..., 0., 0., 8.],
       [2., 0., 0., ..., 0., 0., 8.],
       [2., 0., 0., ..., 0., 0., 8.],
       ...,
       [2., 0., 0., ..., 0., 0., 8.],
       [2., 0., 0., ..., 0., 0., 8.],
       [2., 0., 0., ..., 0., 0., 8.]], dtype=float32)

In [ ]:
3898846 + 385600

4284446

In [46]:
y_probability_all

array([[0.8344414 , 0.16555855],
       [0.7431937 , 0.25680625],
       [0.982463  , 0.01753706],
       ...,
       [0.85561293, 0.14438707],
       [0.88063365, 0.11936634],
       [0.8069839 , 0.19301614]], dtype=float32)

In [ ]:
3898846 + 385600

4284446

In [ ]:
3898846 + 385600

4284446

In [36]:
y_prediction.shape

(77120,)

In [ ]:
3898846 + 385600

4284446

In [ ]:
3898846 + 385600

4284446

In [31]:
csfs_above_threshold_indices.shape

(26971,)

In [30]:
ml_chosen_from_unselected_indices.shape

(32194,)

In [32]:
zxcvasdf = np.union1d(csfs_above_threshold_indices, ml_chosen_from_unselected_indices)

In [34]:
zxcvasdf.shape

(58302,)

In [23]:
rmix_file_data.block_CSFs_nums

[np.int32(385600)]

In [25]:
raw_csfs_descriptors.shape

(4284446, 87)

In [22]:
len(y_unselected_probability)

3898846

In [26]:
3898846 + 385600

4284446

In [21]:
y_unselected_probability

array([0.20182517, 0.20514992, 0.2887378 , ..., 0.03118911, 0.04493222,
       0.16360037], dtype=float32)

In [20]:
ml_chosen_from_unselected_indices

array([], dtype=int64)

In [17]:
y_prediction

array([False, False, False, ..., False, False, False])

In [18]:
np.where(y_prediction == 1)[0]

array([    9,    23,    35, ..., 77080, 77111, 77115])

In [24]:
y_stay_pred = model.predict(X_stay)

In [40]:
ml_predict_index = np.where(y_stay_pred == 1)[0]

In [32]:
y_stay_proba = model.predict_proba(X_stay)[:, 1]

In [34]:
y_stay_proba.shape

(3898846,)

In [36]:
y_stay_pred.shape

(3898846,)

In [37]:
target_pool_csfs_data.CSFs_block_length

array([4284446])

In [38]:
cal_csfs_data.CSFs_block_length

array([385600])

In [41]:
ml_predict_index.shape

(30679,)

In [42]:
rmix_file_data.mix_coefficient_List

[array([[ 3.88368822e-01, -3.75787854e-01,  2.86599207e-01, ...,
          1.36427027e-07,  6.39849076e-07,  5.39727752e-08],
        [ 1.24814669e-02, -1.21756193e-02,  9.17219688e-03, ...,
         -6.31284283e-07,  5.70214263e-07,  3.64236044e-07]])]

In [45]:
mix_coeff_above_threshold = gdp.batch_asfs_mix_square_above_threshold(rmix_file_data, threshold=config.cutoff_value)

In [49]:
mix_coeff_above_threshold[0].shape

(26971,)

In [50]:
selected_indices_file = Path(config.root_path) / f"{config.conf}_selected_indices.pkl"
selected_csfs_indices_dict = gdp.csfs_index_load(selected_indices_file)

In [47]:
llasdg = np.union1d(ml_predict_index, mix_coeff_above_threshold[0])

In [48]:
llasdg.shape

(56594,)

In [54]:
energy_level_data_pd['EnergyTotal'][0]

np.float64(-11274.7413433)